# 01 — Fetch Event Option Chains

For every S&P 500 earnings event in our 2021-06-23 → 2026-04-21 window we need
the option chain on **two specific dates**:

- `pre_date` = 1 trading day before the announcement → gives us the market's
  *expected* move (`straddle_pct_pre`) and the IV going into earnings
- `post_date` = 1 trading day after the announcement → gives us the realized
  spot price (so we can measure `actual_move_pct`) and the IV after the crush

**Source**: Alpha Vantage `HISTORICAL_OPTIONS` (premium tier, 75 req/min).

**Total budget**: ~8,000 events × 2 dates ≈ **16,000 calls** ≈ 4-7 hours wall-clock
at the actual ~0.6 calls/sec we measured (HTTP latency dominates).

**Storage**: ~4.6 GB of JSON cached under `unified_strategy/cache/options/{TICKER}/{date}.json`.
Each chain has every strike × every expiration × calls + puts (typically
2,000-6,000 contracts per file).

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

from unified_strategy import CACHE_DIR
from unified_strategy.features import load_all_stock_events, load_sp500_tickers

## 1. Build the event list

Each row is one earnings announcement. The (ticker, announcement_date) pair
determines the two API calls we make.

In [2]:
tickers = load_sp500_tickers()
events = load_all_stock_events(tickers)
print(f"SP500 events in window: {len(events):,}")
print(f"Unique tickers: {events['ticker'].nunique()}")
print(f"Date range: {events['announcement_date'].min().date()} → {events['announcement_date'].max().date()}")
events.head(3)

SP500 events in window: 8,005
Unique tickers: 485
Date range: 2021-06-24 → 2025-12-02


,ticker,fiscal_quarter_end,announcement_date,eps_beat,price_1m_before,price_3m_before,price_change_1m_pct,price_change_3m_pct,eps_estimate_average,eps_estimate_high,...,elo_vol_4q,total_revenue_yoy_growth_lag1,total_revenue_qoq_growth_lag1,total_revenue_ttm_yoy_growth_lag1,actual_eps_yoy_growth_lag1,actual_eps_qoq_growth_lag1,ebitda_yoy_growth_lag1,operating_income_yoy_growth_lag1,gross_margin_yoy_change_lag1,operating_margin_yoy_change_lag1
0,ACN,2021-05-31,2021-06-24,1.0,285.99,280.77,1.95,3.84,2.23,2.30,...,525.45,8.50,NaN,3.69,6.28,25.31,15.26,11.05,-0.41,0.32
1,CCL,2021-05-31,2021-06-28,0.0,29.56,26.65,-11.54,-1.88,-1.61,-1.01,...,1955.08,-99.46,NaN,-95.07,-913.64,11.39,-609.66,-113.74,-1984.14,-5846.65
2,FDS,2021-05-31,2021-06-29,0.0,334.36,308.59,0.52,8.92,2.74,2.88,...,411.14,5.95,NaN,4.82,6.67,5.43,18.66,9.29,-2.26,0.90


## 2. Cache audit (already-fetched chains)

The fetcher is idempotent — already-cached chains are skipped. Below counts
how many JSON files we already have, broken down by ticker.

In [3]:
cached = {}
for d in CACHE_DIR.iterdir() if CACHE_DIR.exists() else []:
    if d.is_dir():
        cached[d.name] = len(list(d.glob("*.json")))

cached_df = pd.Series(cached).sort_values(ascending=False)
total = cached_df.sum()
print(f"Total cached chains: {total:,}")
print(f"Tickers with cache: {len(cached_df)}")
print()
print("Top 10 tickers by cache size:")
print(cached_df.head(10).to_string())

Total cached chains: 16,011
Tickers with cache: 485

Top 10 tickers by cache size:
JPM     37
F       36
KMB     36
ABT     36
SYF     36
EVRG    36
PCAR    36
MDLZ    36
CMS     36
YUM     36


## 3. The fetcher in production

We use `run_bulk_fetch.py` for the actual scrape. It:

1. Detects rate-limit responses (`Information` / `Note` keys) and **does not**
   cache them — sleeps 60s then retries indefinitely
2. 3× exponential backoff on transient network errors (1s → 2s → 4s)
3. Skips cached files instantly on re-run
4. Flushes a tracking CSV every 250 events for crash recovery

In our actual run it survived two ISP outages with **0.18% loss** that a
30-second second-pass cleaned up perfectly.

Run it from the shell:

```bash
export ALPHAVANTAGE_API_KEY=...
python3 run_bulk_fetch.py
```

## 4. Sample a chain to confirm schema

In [4]:
import json

# Pick the first cached chain we can find
sample_path = next(CACHE_DIR.rglob("*.json"))
data = json.loads(sample_path.read_text())
print(f"Sample file: {sample_path.relative_to(CACHE_DIR.parent)}")
print(f"Top-level keys: {list(data.keys())}")
contracts = data.get("data", [])
print(f"Contracts in this chain: {len(contracts):,}")
print()
print("Schema of one contract:")
if contracts:
    sample = contracts[0]
    for k, v in sample.items():
        print(f"  {k}: {v}")

Sample file: options/CTAS/2024-12-20.json
Top-level keys: ['endpoint', 'message', 'data']
Contracts in this chain: 1,252

Schema of one contract:
  contractID: CTAS241220C00075000
  symbol: CTAS
  expiration: 2024-12-20
  strike: 75.00
  type: call
  last: 0.00
  mark: 112.00
  bid: 107.00
  bid_size: 3
  ask: 117.00
  ask_size: 3
  volume: 0
  open_interest: 0
  date: 2024-12-20
  implied_volatility: 6.60008
  delta: 0.99757
  gamma: 0.00012
  theta: -0.25268
  vega: 0.00074
  rho: 0.00204


## 5. Final coverage stats

In [5]:
n_events = len(events)
expected_calls = n_events * 2
cached_now = sum(cached.values())
print(f"Events:          {n_events:,}")
print(f"Calls expected:  {expected_calls:,}")
print(f"Cached:          {cached_now:,} ({100 * cached_now / expected_calls:.1f}%)")
print()
print("If coverage is 99%+, we're ready for notebook 02 (features).")

Events:          8,005
Calls expected:  16,010
Cached:          16,011 (100.0%)

If coverage is 99%+, we're ready for notebook 02 (features).
